Data Collection from HSP API

In [18]:
#Import necessary libraries 
import requests
import json
import base64
import pandas as pd

#Define the API endpoints
metrics_url = 'https://hsp-prod.rockshore.net/api/v1/serviceMetrics'
details_url = 'https://hsp-prod.rockshore.net/api/v1/serviceDetails'

#Login details for API access
email = 'shanelleferns3@gmail.com'
password = '~5GLz%d8bjJ-c+k'

#Combine email and password and encode it
credentials = f'{email}:{password}'
encoded_credentials = base64.b64encode(credentials.encode()).decode()

#Define the request headers
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Basic {encoded_credentials}'
}

#Define the body for the metrics API
metrics_body = {
    "from_loc": "LIV",
    "to_loc": "BPN",
    "from_time": "0000",
    "to_time": "2359",
    "from_date": "2024-01-01",
    "to_date": "2024-07-31",
    "days": "WEEKDAY"  
}

#Make the POST request to the metrics API
response = requests.post(metrics_url, headers=headers, data=json.dumps(metrics_body))

#Check if the request was successful
if response.status_code == 200:
    #Parse the JSON response
    metrics_data = response.json()

    #Initialize a list to hold results
    combined_results = []

    #Iterate over each service in the metrics response
    for service in metrics_data.get('Services', []):
        service_metrics = service.get('serviceAttributesMetrics', {})
        rids = service_metrics.get('rids', [])

        #Iterate over each RID and fetch the details
        for rid in rids:
            #Define the body for the details API
            details_body = {
                "rid": rid
            }

            #Make the POST request to the details API
            details_response = requests.post(details_url, headers=headers, data=json.dumps(details_body))

            #Check if the request was successful
            if details_response.status_code == 200:
                #Parse the JSON response
                details_data = details_response.json()

                #Combine the metrics and details data
                combined_result = {
                    "service_metrics": service_metrics,
                    "service_details": details_data.get('serviceAttributesDetails', {})
                }

                #Add the combined result to the list
                combined_results.append(combined_result)
            else:
                print(f'Failed to fetch details for RID {rid}. HTTP Status code: {details_response.status_code}')
                print(details_response.text)

    #Create a DataFrame from the combined results
    records = []
    for result in combined_results:
        metrics = result["service_metrics"]
        details = result["service_details"]

        for location in details.get("locations", []):
            record = {
                "origin_location": metrics.get("origin_location"),
                "destination_location": metrics.get("destination_location"),
                "gbtt_ptd": metrics.get("gbtt_ptd"),
                "gbtt_pta": metrics.get("gbtt_pta"),
                "toc_code": metrics.get("toc_code"),
                "matched_services": metrics.get("matched_services"),
                "rid": details.get("rid"),
                "date_of_service": details.get("date_of_service"),
                "location": location.get("location"),
                "gbtt_ptd_location": location.get("gbtt_ptd"),
                "gbtt_pta_location": location.get("gbtt_pta"),
                "actual_td_location": location.get("actual_td"),
                "actual_ta_location": location.get("actual_ta"),
                "late_canc_reason": location.get("late_canc_reason")
            }
            records.append(record)

    df = pd.DataFrame(records)
    print(df)
else:
    print(f'Failed to fetch metrics data. HTTP Status code: {response.status_code}')
    print(response.text)


      origin_location destination_location gbtt_ptd gbtt_pta toc_code  \
0                 LIV                  BPN     0638     0757       NT   
1                 LIV                  BPN     0638     0757       NT   
2                 LIV                  BPN     0638     0757       NT   
3                 LIV                  BPN     0638     0757       NT   
4                 LIV                  BPN     0638     0757       NT   
...               ...                  ...      ...      ...      ...   
27425             LIV                  BPN     2215     2347       NT   
27426             LIV                  BPN     2215     2347       NT   
27427             LIV                  BPN     2215     2347       NT   
27428             LIV                  BPN     2215     2347       NT   
27429             LIV                  BPN     2215     2347       NT   

      matched_services              rid date_of_service location  \
0                  147  202401018730006      2024-01-01

In [19]:
#Save the DataFrame as a CSV file
df.to_csv('data.csv', index=False)